# Autopatch demo performance

Analyze the saved MultiPatch logs from an extended autopatch demo run: the
find → seal → break-in funnel, throughput/efficiency, where time goes, why
attempts fail, and the electrical quality of the cells that made it to whole-cell.

Metrics are reconstructed from the `state_change` sequence and `test_pulse`
events in each `MultiPatch_*.log` (the tidy `patchRecord` is emitted as a Qt
signal, not written to the log). Parsing/aggregation live in `autopatch_log.py`
and `autopatch_metrics.py` (unit-tested under `tests/`).

## 1. Configure & load

Point `ROOTS` at the directory (or directories) holding the demo's
`MultiPatch_*.log` files. They're found recursively, one attempt-set per file.

The analysis is restricted to attempts that actually approached a cell
(`ONLY_APPROACHED`); setup/cleaning cycles that never engaged a cell are
excluded so the funnel and active-time reflect only real patch attempts.

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath('.'))
import autopatch_log as al
import autopatch_metrics as am
import autopatch_session_log as asl

# ---- EDIT ME: directories containing the demo's MultiPatch_*.log files ----
# One extended-demo run per root works best (throughput assumes contiguous time).
ROOTS = [
    "C:\\data\\martin\\2026.09.04_000",
    "C:\\data\\martin\\2026.09.03_000",
    "C:\\data\\martin\\2026.09.02_000",
]

# Restrict the analysis to attempts that actually engaged a cell (reached the
# approach stage). Attempts that never got past bath/clean/out are pipette
# setup and cleaning cycles, not real patch attempts -- keeping them would
# inflate the funnel base and pad the active time with pre/post-demo idle.
ONLY_APPROACHED = True

# Confidence level for the Wilson binomial intervals on the funnel and yield.
CI_LEVEL = 0.95

# A stretch this long (minutes) with no device patching, cleaning or holding a
# recording counts as autopatch stopped, not as the rhythm between cells.
IDLE_MIN_MINUTES = 10.0

plt.rcParams.update({'figure.figsize': (9, 4.5), 'axes.grid': True,
                     'grid.alpha': 0.25, 'axes.axisbelow': True})
# Consistent funnel palette (dark→light single hue reads as a progression).
FUNNEL_COLORS = ['#1a4a73', '#2d6ca8', '#4a90d9', '#8bc0f0', '#2e8b57']
OK, BAD, IDLE = '#2e8b57', '#c0392b', '#e8973a'

def si(val, unit):
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return 'n/a'
    for p, s in [(1e9, 'G'), (1e6, 'M'), (1e3, 'k'), (1, ''), (1e-3, 'm'), (1e-6, 'µ'), (1e-9, 'n'), (1e-12, 'p')]:
        if abs(val) >= p:
            return f'{val / p:.2f} {s}{unit}'
    return f'{val:.2e} {unit}'

def pct_ci(lo, hi):
    """Compact '(lo–hi)' label for a percentage interval; empty when undefined."""
    if not (np.isfinite(lo) and np.isfinite(hi)):
        return ''
    return f'({lo:.0f}–{hi:.0f})'

In [ ]:
# raw_attempts keeps every attempt (incl. setup/cleaning idle) for the timeline;
# `attempts` is the analysis population -- approached attempts only by default.
raw_attempts = al.load_run(ROOTS)
attempts = al.approached_attempts(raw_attempts) if ONLY_APPROACHED else raw_attempts
df = am.attempts_to_dataframe(attempts)
assert len(df), 'No approached attempts found. Check ROOTS, or set ONLY_APPROACHED=False.'
n_excluded = len(raw_attempts) - len(attempts)
print(f'{len(al.find_logs(ROOTS))} log files, {len(raw_attempts)} raw attempts, '
      f'{len(df)} approached (analyzed), {n_excluded} setup/cleaning excluded, '
      f"{df['device'].nunique()} device(s)")
# An attempt whose log stops in a state the pipette only passes through never
# recorded how it ended, so every count below is a lower bound by that many.
# acq4 used to truncate a log this way whenever the recorder was torn down
# before the terminal state_change had been written.
unlogged = df[~df['outcome_recorded']]
if len(unlogged):
    print(f'\nWARNING: {len(unlogged)} attempt(s) have no recorded outcome; their log '
          f'stops mid-attempt:')
    print(unlogged[['cell_dir', 'final_state', 'best_stage_name']].to_string(index=False))
# Cleaning, idle and the stopped-time shading in §5 come from the ACQ4 session
# log: the MultiPatch recorder is not running between attempts.
session_spans, surveys = asl.load_session_logs(ROOTS)
print(f'{len(session_spans)} session-log state span(s)')
df.head()

## 2. Data consistency

A functioning Autopatch run opens a cell directory per patch attempt, so each
directory should hold exactly one approached attempt working exactly one cell.
Everything downstream counts attempt records, so that contract is what makes
the funnel a count of cells. This checks it before any of the numbers lean on
it.

Every count below except `n_cleaning` should read 1. Anything else is flagged:
several attempt records in one directory, none at all, more than one sealing
episode, targets more than `DISTINCT_TARGET_DISTANCE` apart, or more than one
whole-cell recording. Each of those means several cells landed in one folder,
and the funnel reads high by exactly that much — one directory holding a
failure and a success scores as one success.

Logs recorded by hand (record toggled on once for a whole session rather than
per cell) light up every row. That is the check working: those runs are not
Autopatch runs, and their funnel percentages are per recording stretch, not
per cell.

In [ ]:
cons = am.consistency_report(raw_attempts)
n_ok = int(cons['ok'].sum())
print(f'{n_ok} of {len(cons)} cell director{"y" if len(cons) == 1 else "ies"} '
      f'hold exactly one attempt on one cell.')
if n_ok < len(cons):
    print('\nFlagged — the funnel below counts attempt records, so it reads high here:')
    for _, r in cons[~cons['ok']].iterrows():
        print(f"  {r['location']:>28s}  {r['device']:14s} {r['anomalies']}")
cons.drop(columns=['folder'])

## 3. Headline throughput & efficiency

In [ ]:
tp = am.throughput(df, confidence=CI_LEVEL)
print('Logs                :', int(tp['n_logs']))
print('Approached attempts :', int(tp['n_attempts']))
print('Cells found         :', int(tp['n_found']))
print('Seals formed        :', int(tp['n_sealed']))
print('Whole-cell recordings:', int(tp['n_whole_cell']),
      f"(+{int(tp['n_outcome_unlogged'])} attempt(s) whose outcome was never logged)"
      if tp['n_outcome_unlogged'] else '')
print('Overall yield       : {:.1f} % {} at {:.0f}% CI  (whole-cell / approached)'.format(
    tp['overall_yield_pct'], pct_ci(tp['overall_yield_lo_pct'], tp['overall_yield_hi_pct']), 100 * CI_LEVEL))
print('Active time         : {:.2f} h  (demo window, summed per-log)'.format(tp['active_hours']))
print('Attempts / hour     : {:.1f}'.format(tp['attempts_per_hour']))
print('Whole-cells / hour  : {:.2f}'.format(tp['whole_cells_per_hour']))
print('Mean attempt length : {:.1f} min'.format(tp['mean_attempt_minutes']))

## 4. The funnel: approached → found → sealed → whole-cell

How many *approached* attempts survive each stage, and the conversion rate
between stages. Attempts that never engaged a cell (setup/cleaning cycles) are
excluded, so the base of the funnel is the cells we actually tried to patch.

Error bars and bracketed ranges are Wilson score intervals on each proportion
(95% by default — see `CI_LEVEL`). A run is a few dozen attempts at most, so
they are wide: read `54% (28–79)` as "somewhere between a quarter and four
fifths", not as a number to compare against another run's second decimal. They
also assume attempts are independent trials, which is optimistic — attempts
sharing a slice, a pipette and a software version fail together — and they say
nothing about the bias section 2 measures.

In [ ]:
funnel = am.funnel_counts(df, confidence=CI_LEVEL)
labels = ['Approached', 'Found\ncell', 'Sealed', 'Whole\ncell']
colors = [FUNNEL_COLORS[1], FUNNEL_COLORS[2], FUNNEL_COLORS[3], FUNNEL_COLORS[4]]
# The intervals are on proportions of the funnel base; draw them in the bars' units.
n_approached = int(funnel['count'].iloc[0])
lo_count = funnel['pct_of_approached_lo'] / 100.0 * n_approached
hi_count = funnel['pct_of_approached_hi'] / 100.0 * n_approached

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(labels, funnel['count'], color=colors, width=0.72)
# offset right of centre so the whiskers miss the conversion label inside the bar
err_x = [bar.get_x() + bar.get_width() * 0.8 for bar in bars]
ax.errorbar(err_x, funnel['count'],
            yerr=[funnel['count'] - lo_count, hi_count - funnel['count']],
            fmt='none', ecolor='0.2', capsize=6, lw=1.4, zorder=4)
for i, (bar, (_, row)) in enumerate(zip(bars, funnel.iterrows())):
    x = bar.get_x() + bar.get_width() / 2
    # the base stage is 100% by definition, so it gets no interval
    ci = '' if i == 0 else ' ' + pct_ci(row['pct_of_approached_lo'], row['pct_of_approached_hi'])
    ax.annotate(f"{int(row['count'])}\n{row['pct_of_approached']:.0f}%{ci}",
                xy=(x, hi_count.iloc[i]), xytext=(0, 5), textcoords='offset points',
                ha='center', va='bottom', fontsize=10)
    if i:
        conv = pct_ci(row['conversion_from_prev_lo'], row['conversion_from_prev_hi'])
        ax.text(x, bar.get_height() / 2, f"→{row['conversion_from_prev']:.0f}%\n{conv}",
                ha='center', va='center', color='white', fontsize=8.5, fontweight='bold')
ax.set_ylabel('approached attempts')
ax.set_title('Autopatch funnel (bar height = count ± {:.0f}% CI, mid-bar = conversion from previous stage)'.format(100 * CI_LEVEL))
ax.margins(y=0.2)
plt.tight_layout(); plt.show()
funnel.round(1)

## 5. Throughput over the run

Cumulative whole-cell recordings against elapsed time — the slope is the
effective yield rate.

Shaded spans are stretches of at least `IDLE_MIN_MINUTES` in which no device was
patching, cleaning or holding a recording: autopatch stopped rather than
working. A flat stretch of the curve inside a shaded span means "not running";
a flat stretch outside one means "running and failing". The ordinary rhythm
between cells is deliberately left unshaded — lower `IDLE_MIN_MINUTES` to see
it.

Like section 6b this reads the session log, because the MultiPatch recorder is
not running between attempts; without a `log.json` under `ROOTS` the curve is
drawn unshaded. With several days under `ROOTS` the overnight gaps shade too —
correctly, but they dominate the axis, and section 11 is the better view there.

In [ ]:
cum = am.cumulative_whole_cells(df)
t0 = df['start_time'].min()  # the zero of cum['minutes']
x_max = (df['end_time'].max() - t0) / 60.0
idle = am.session_idle_spans(session_spans, min_minutes=IDLE_MIN_MINUTES)

fig, ax = plt.subplots()
for i, (_, r) in enumerate(idle.iterrows()):
    ax.axvspan((r['t_start'] - t0) / 60.0, (r['t_end'] - t0) / 60.0,
               color=IDLE, alpha=0.3, lw=0, zorder=0,
               label=f'autopatch stopped (≥{IDLE_MIN_MINUTES:.0f} min)' if not i else None)
if len(cum):
    # anchored at the origin so the lead-in before the first success is visible
    ax.step([0] + list(cum['minutes']), [0] + list(cum['cumulative_whole_cells']),
            where='post', color=OK, lw=2)
    ax.scatter(cum['minutes'], cum['cumulative_whole_cells'], color=OK, zorder=3)
ax.set_xlim(0, x_max * 1.02)
ax.set_xlabel('minutes since first attempt')
ax.set_ylabel('cumulative whole-cell recordings')
ax.set_title('Whole-cell yield over the run')
if len(idle):
    ax.legend(loc='upper left', fontsize=9)
plt.tight_layout(); plt.show()

if not session_spans:
    print('No session log under ROOTS, so stopped time is not shaded here (see §6b).')
elif len(idle):
    logged_min = (max(s.t_end for s in session_spans)
                  - min(s.t_start for s in session_spans)) / 60.0
    print('Stopped for {:.0f} of the session log\'s {:.0f} min ({:.0f} %):'.format(
        idle['minutes'].sum(), logged_min, 100 * idle['minutes'].sum() / logged_min))
    for _, r in idle.iterrows():
        print('  {:7.1f} → {:7.1f} min  ({:.0f} min)'.format(
            (r['t_start'] - t0) / 60.0, (r['t_end'] - t0) / 60.0, r['minutes']))
else:
    print(f'No stretch of ≥ {IDLE_MIN_MINUTES:.0f} min without a device working.')

## 6. Where the time goes

Per-attempt duration, and how long each patch state takes for the attempts that
reach it (`n` on each bar is how many that was).

`total_s` sums over every attempt, but `mean_s` averages only over
`n_attempts` — the attempts that actually reached that state. If two attempts
made it to `cell attached`, its mean divides by two. `n_visits` counts state
intervals, so it exceeds `n_attempts` where attempts re-entered a state; all of
one attempt's visits count as a single value in the mean.

The state an attempt ended in is excluded: the log records it being entered but
never left, so its span is just time-until-the-log-stopped, not a dwell time.
That drops the terminal states (`whole cell`, `fouled`); the `outcome` column
in section 10 says where each attempt stopped.

In [ ]:
dwell = am.state_dwell_times(attempts)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

ax1.hist(df['duration_s'] / 60.0, bins=20, color=FUNNEL_COLORS[1], edgecolor='white')
ax1.set_xlabel('attempt duration (min)'); ax1.set_ylabel('attempts')
ax1.set_title('Per-attempt duration')

by_mean = dwell.sort_values('mean_s', ascending=False)
bars = ax2.barh(by_mean['state'], by_mean['mean_s'], color=FUNNEL_COLORS[2])
for bar, n in zip(bars, by_mean['n_attempts']):
    ax2.text(bar.get_width(), bar.get_y() + bar.get_height() / 2, f' n={int(n)}',
             va='center', fontsize=8, color='0.35')
ax2.invert_yaxis()
ax2.margins(x=0.12)
ax2.set_xlabel('mean seconds in state, per attempt that reached it')
ax2.set_title('How long each state takes')
plt.tight_layout(); plt.show()
dwell

### 6b. The rest of the clock: cleaning, cell detection, and idle

The states above come from the MultiPatch logs, and those are only recorded while
a cell directory is open — that is, only while the pipette is patching. Cleaning
the tip, surveying a tile for the next cells, and parking between attempts all
happen with no recorder attached, so none of them appear above. On an extended
run they are about half the clock.

ACQ4's own session log (`log.json`) does cover them, so this section reads from
there instead (`autopatch_session_log.py`):

- **cleaning** — the `clean`/`blowout` states: the trip to the well, sonication, and the trip back.
- **cell detection** — each `tile_detector` survey: scope move, z-stack, detection model.
- **idle** — `out`/`bath`/`fouled`/`broken`: parked, travelling, or waiting for the queue to pick the next cell.
- **patching** — approach → contact → seal → break-in. Shown as a cross-check: it should match the MultiPatch active time above.
- **recording** — `whole cell` and after. Its length is set by the experiment's task protocol, so it is reported but never counted as overhead.

The window is trimmed to first approach → last clean/recording, so the setup
before the run and the idle after it are not charged to it. Detection is reported
*separately* rather than as a slice of the budget: a survey runs concurrently with
whatever state the pipette is in, so adding it to the categories would
double-count (`blocking_minutes` is how much of it overlapped non-patching time).


In [ ]:
budget = am.session_time_budget(session_spans)

if budget.empty:
    print('No ACQ4 session log (log.json) found under ROOTS, or it recorded no patch states.\n'
          'Cleaning / detection / idle time cannot be recovered from MultiPatch logs alone:\n'
          'point ROOTS at the run directory ACQ4 wrote log.json into.')
else:
    cats = am.session_category_totals(budget)
    surv = am.survey_time(session_spans, surveys)
    summary = am.session_time_summary(session_spans, surveys, n_attempts=len(df))

    print('Active window        : {:.1f} min  (first approach -> last clean, wall-clock)'.format(summary['active_minutes']))
    print('  patch attempts     : {:.1f} min ({:.0f} %)'.format(summary['patching_minutes'], 100 * summary['patching_minutes'] / summary['active_minutes']))
    print('  cleaning           : {:.1f} min ({:.0f} %) over {:d} cycles, {:.0f} s each'.format(
        summary['cleaning_minutes'], 100 * summary['cleaning_minutes'] / summary['active_minutes'],
        int(summary['n_clean_cycles']),
        60 * summary['cleaning_minutes'] / max(int(summary['n_clean_cycles']), 1)))
    print('  idle / travelling  : {:.1f} min ({:.0f} %)'.format(summary['idle_minutes'], 100 * summary['idle_minutes'] / summary['active_minutes']))
    print('  whole-cell holding : {:.1f} min ({:.0f} %)  <- task protocol, not counted as overhead'.format(
        summary['recording_minutes'], 100 * summary['recording_minutes'] / summary['active_minutes']))
    print('Non-patching overhead: {:.1f} min ({:.0f} % of the run)'.format(summary['overhead_minutes'], summary['overhead_pct']))
    if len(surv):
        print('Cell detection       : {:.1f} min ({:.0f} %) over {:d} tile survey(s), {:.0f} s each, '
              '{:d} candidates ({:.1f} s/candidate)'.format(
                  surv['minutes'], surv['pct_of_active'], int(surv['n_surveys']),
                  60 * surv['mean_minutes'], int(surv['n_candidates']), surv['per_candidate_s']))
        print('                       {:.1f} min of that ran while the pipette was not patching'.format(surv['blocking_minutes']))
    else:
        print('Cell detection       : no tile surveys logged in the window')
    print('Per attempt          : {:.2f} min total, of which {:.2f} min cleaning, {:.2f} min idle, {:.2f} min detection'.format(
        summary['minutes_per_attempt'], summary['cleaning_per_attempt_minutes'],
        summary['idle_per_attempt_minutes'], summary['detection_per_attempt_minutes']))

    CATEGORY_COLORS = {'patching': FUNNEL_COLORS[2], 'cleaning': '#d98c2b',
                       'idle': '0.7', 'recording': OK, 'other': BAD}
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5),
                                   gridspec_kw={'height_ratios': [1]})

    # Left: the run's wall-clock split into categories, with the concurrent
    # detection cost drawn underneath rather than stacked into it.
    left = 0.0
    for _, row in cats.iterrows():
        ax1.barh(1, row['minutes'], left=left, color=CATEGORY_COLORS.get(row['category'], BAD),
                 edgecolor='white', height=0.5, label=row['category'])
        if row['pct_of_active'] >= 4:
            ax1.text(left + row['minutes'] / 2, 1, '{:.0f} %'.format(row['pct_of_active']),
                     ha='center', va='center', color='white', fontweight='bold', fontsize=9)
        left += row['minutes']
    if len(surv) and surv['minutes'] > 0:
        ax1.barh(0.45, surv['minutes'], color='none', edgecolor=FUNNEL_COLORS[0],
                 hatch='///', height=0.22, label='cell detection (concurrent)')
        ax1.text(surv['minutes'] + left * 0.01, 0.45, ' detection {:.1f} min'.format(surv['minutes']),
                 va='center', fontsize=8, color=FUNNEL_COLORS[0])
    ax1.set_ylim(0.2, 1.45)
    ax1.set_yticks([])
    ax1.set_xlabel('minutes')
    ax1.set_title('Where the run\'s wall-clock went ({:.0f} min)'.format(summary['active_minutes']))
    ax1.legend(loc='upper center', bbox_to_anchor=(0.5, -0.18), ncol=3, fontsize=8, frameon=False)

    # Right: the same total, per state.
    by_total = budget.sort_values('total_s')
    ax2.barh(by_total['state'], by_total['total_s'] / 60.0,
             color=[CATEGORY_COLORS.get(c, BAD) for c in by_total['category']])
    for y, (_, row) in enumerate(by_total.iterrows()):
        ax2.text(row['total_s'] / 60.0, y, ' n={:d}, {:.0f} s each'.format(int(row['n_visits']), row['mean_s']),
                 va='center', fontsize=8, color='0.35')
    ax2.margins(x=0.28)
    ax2.set_xlabel('total minutes over the run')
    ax2.set_title('Total time per state (session log)')
    plt.tight_layout(); plt.show()

budget

## 7. How long to keep trying to seal

Time in the `seal` state -- suction on a detected cell -- against whether that
episode went on to seal and to a recording. This is the stretch
`SealState.autoSealTimeout` caps (default 30 s, in
`acq4/devices/PatchPipette/states/seal.py`); `SEAL_CUTOFF` below is the value to
play with.

- **left**: how long sealing took, split by what it won. The mode is the cells
  that seal quickly; the tail is what a shorter timeout gives up.
- **middle**: of the episodes still sealing at *t*, the share that went on to
  seal -- the odds on one more second of waiting. The grey line is how many
  episodes are left to judge it on, which is what makes the right-hand end of
  the curve noise.
- **right**: the replay. Whole cells retained and sealing time spent at each
  cutoff.

The replay shortens episodes and cannot lengthen them: an episode the rig gave
up on says nothing about what a longer timeout would have won from it. Past the
longest episode on record (shaded) every row is a floor on the cost and a
ceiling on the yield, not an estimate. Episodes a log stops inside are dropped,
since their outcome is unknown rather than failed.

`whole_cells_per_seal_hour` counts sealing time only, so it always favours a
short cutoff. Giving up buys a fresh cell at the price of another approach,
clean and detection -- the per-attempt overhead in section 6b -- so compare
`minutes_saved` against that, not against zero.

In [ ]:
# ---- EDIT ME: candidate autoSealTimeout (seconds) ----
SEAL_CUTOFF = 30.0

seals = am.seal_episode_table(attempts)
live = seals[~seals['censored']]
print(f'{len(seals)} seal episodes '
      f"({int(seals['censored'].sum())} cut off by the end of a log, dropped), "
      f"{int(live['sealed'].sum())} sealed, "
      f"{int(live['whole_cell'].sum())} went on to whole cell")

if live.empty:
    print('No completed seal episodes: nothing to tune.')
else:
    # Replay out to whichever is further, the longest episode or the cutoff
    # being tried, so a cutoff past the data is drawn past the data.
    observed_max = live['seconds'].max()
    sweep = am.seal_cutoff_sweep(
        seals, cutoffs=np.arange(1.0, max(observed_max, SEAL_CUTOFF) + 1.0))
    within = sweep[sweep['within_data']]
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 4.3))

    groups = [('whole cell', live['whole_cell'], OK),
              ('sealed only', live['sealed'] & ~live['whole_cell'], FUNNEL_COLORS[3]),
              ('gave up', ~live['sealed'], BAD)]
    bins = np.histogram_bin_edges(live['seconds'], bins=min(30, max(8, len(live) // 3)))
    ax1.hist([live.loc[mask, 'seconds'] for _, mask, _ in groups], bins=bins,
             stacked=True, color=[c for _, _, c in groups],
             label=[name for name, _, _ in groups], edgecolor='white', linewidth=0.5)
    ax1.axvline(SEAL_CUTOFF, color='0.3', ls='--', lw=1)
    ax1.legend(fontsize=8)
    ax1.set_xlabel('seconds in the seal state'); ax1.set_ylabel('episodes')
    ax1.set_title('How long sealing took')

    ax2.plot(within['cutoff_s'], within['pct_of_cut_that_sealed'], color=FUNNEL_COLORS[1], lw=2)
    ax2.axvline(SEAL_CUTOFF, color='0.3', ls='--', lw=1)
    ax2.set_ylim(0, 100)
    ax2.set_xlabel('seconds already spent sealing')
    ax2.set_ylabel('% that still seal', color=FUNNEL_COLORS[1])
    ax2.set_title('Odds on waiting longer')
    ax2b = ax2.twinx()
    ax2b.step(within['cutoff_s'], within['n_cut'], where='post', color='0.6', lw=1)
    ax2b.set_ylabel('episodes still sealing', color='0.5')
    ax2b.grid(False)

    ax3.plot(sweep['cutoff_s'], sweep['pct_whole_cells_kept'], color=OK, lw=2)
    if SEAL_CUTOFF > observed_max:
        ax3.axvspan(observed_max, sweep['cutoff_s'].max(), color='0.5', alpha=0.12,
                    zorder=0, label='beyond the data')
        ax3.legend(fontsize=8, loc='lower right')
    ax3.axvline(SEAL_CUTOFF, color='0.3', ls='--', lw=1)
    ax3.set_ylim(0, 103)
    ax3.set_xlabel('cutoff (s)')
    ax3.set_ylabel('% of whole cells kept', color=OK)
    ax3.set_title('What each cutoff keeps and costs')
    ax3b = ax3.twinx()
    ax3b.plot(sweep['cutoff_s'], sweep['seal_minutes'], color=FUNNEL_COLORS[1], lw=1.5, ls=':')
    ax3b.set_ylabel('minutes spent sealing', color=FUNNEL_COLORS[1])
    ax3b.grid(False)

    plt.tight_layout(); plt.show()

    at = am.seal_cutoff_sweep(seals, cutoffs=[SEAL_CUTOFF]).iloc[0]
    print(f'At a {SEAL_CUTOFF:.0f} s cutoff: keeps '
          f"{int(at['whole_cells_kept'])} whole cells ({at['pct_whole_cells_kept']:.0f}% of those recorded), "
          f"cuts {int(at['n_cut'])} episodes of which {int(at['seals_lost'])} would have sealed, "
          f"spends {at['seal_minutes']:.1f} min sealing (saves {at['minutes_saved']:.1f} min)")
    if not at['within_data']:
        print(f'  ...but no episode ran past {observed_max:.0f} s, so that row is a floor on the '
              'cost and a ceiling on the yield.')

am.seal_cutoff_sweep(seals, cutoffs=[5, 10, 15, 20, 30, 45, 60, 90])

## 8. What the economic early-exit rule would have done

Section 7 asks about a fixed timeout. `SealState` can instead give up on a cell
when continuing costs more than starting over: the projected time to seal
(from the trend in resistance), divided by the odds the cell is still alive,
against the expected wall-clock to a seal on a *fresh* cell,
`(R + p*Ts + (1-p)*Tf) / p`. This section replays the logged test pulses
through that rule to ask what it would have cut, what that would have cost in
whole cells, and what it would have saved.

The replay drives acq4's own `SealAnalysis` rather than a copy of its
arithmetic, so it needs acq4 importable from this kernel (the rest of the
notebook does not). The parameters below are `SealAnalysis` keyword arguments:

- `restart_cost` (R): seconds of overhead to get back to a sealing cell --
  the per-attempt overhead in section 6b.
- `seal_probability` (p): chance a fresh cell seals.
- `expected_seal_duration` / `expected_failed_seal_duration` (Ts / Tf):
  seconds in the seal state when it works / when it does not. Section 7 gives
  both.
- `probability_decay_tau`: how fast the cell's odds fade while sealing; 0 holds
  them constant.
- `min_decision_time`, `slope_tau`: how long to watch before judging, and how
  smooth the trend is.

As in section 7, the replay can only shorten an episode: one the rig already
ended says nothing about what a longer wait would have won. Episodes a log
stops inside are dropped. Always read `pct_of_cut_that_sealed` and
`whole_cells_lost` beside `minutes_saved`: the minutes are only a saving if the
cells cut were not about to seal. Each cut also buys a fresh cell, at the price
of `restart_cost` that this table does not charge.

In [ ]:
# ---- EDIT ME: parameters of the rule to replay (seconds) ----
ECONOMICS = dict(
    restart_cost=120.0,
    seal_probability=0.5,
    expected_seal_duration=15.0,
    expected_failed_seal_duration=30.0,
    probability_decay_tau=0.0,
    min_decision_time=5.0,
    slope_tau=2.0,
)

try:
    replay = am.economic_rule_replay(attempts, **ECONOMICS)
except ImportError as exc:
    replay = None
    print(f'acq4 is not importable from this kernel ({exc}); skipping the replay.')

if replay is not None:
    live = replay[~replay['censored']]
    cut = live[live['fired']]
    print(f'{len(live)} completed seal episodes: the rule would have cut {len(cut)} '
          f"({int(cut['sealed'].sum())} of them went on to seal, "
          f"{int(cut['whole_cell'].sum())} to whole cell), "
          f"saving {live['seconds_saved'].sum() / 60:.1f} min of sealing")

    if len(cut):
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.3))
        groups = [('went on to seal', cut['sealed'], OK), ('would have failed', ~cut['sealed'], BAD)]
        bins = np.histogram_bin_edges(cut['fire_s'], bins=min(20, max(6, len(cut) // 2)))
        ax1.hist([cut.loc[mask, 'fire_s'] for _, mask, _ in groups], bins=bins, stacked=True,
                 color=[c for _, _, c in groups], label=[n for n, _, _ in groups],
                 edgecolor='white', linewidth=0.5)
        ax1.legend(fontsize=8)
        ax1.set_xlabel('seconds into the seal state when the rule fires')
        ax1.set_ylabel('episodes cut')
        ax1.set_title('When it gives up, and on whom')

        ax2.scatter(cut['seconds'], cut['fire_s'], c=[OK if s else BAD for s in cut['sealed']], s=22)
        top = float(cut['seconds'].max())
        ax2.plot([0, top], [0, top], color='0.6', lw=1, ls='--')
        ax2.set_xlabel('seconds the episode actually ran')
        ax2.set_ylabel('seconds when the rule fires')
        ax2.set_title('Each cut episode, against the line of no saving')
        plt.tight_layout(); plt.show()

# Vary the odds of a fresh cell sealing: the one input that is a guess rather than a measurement.
candidates = [dict(ECONOMICS, seal_probability=p) for p in (0.2, 0.35, 0.5, 0.65, 0.8)]
am.economic_rule_sweep(attempts, candidates)[
    ['seal_probability', 'n_episodes', 'n_cut', 'seals_lost', 'whole_cells_lost',
     'pct_of_cut_that_sealed', 'whole_cells_kept', 'pct_whole_cells_kept',
     'seal_minutes', 'minutes_saved']
] if replay is not None else None

## 9. Quality of the whole-cell recordings

For attempts that broke in: peak seal resistance, and access resistance /
holding current during whole-cell (from the test pulses). Lower access
resistance and smaller holding current are healthier.

A run yields a handful of cells, so each panel shows one dot per cell (jittered
vertically) with the median marked; it switches to a histogram once there are
enough cells to bin meaningfully.

In [ ]:
wc = df[df['broke_in']].copy()

def quality_panel(ax, values, xlabel, color):
    """One dot per cell while they are few enough to show individually; histogram beyond that."""
    v = values.dropna().to_numpy(dtype=float)
    if len(v) < 15:
        ax.scatter(v, np.random.default_rng(0).uniform(-0.3, 0.3, len(v)),
                   color=color, alpha=0.85, zorder=3)
        ax.set_ylim(-1, 1); ax.set_yticks([]); ax.set_ylabel(f'{len(v)} cells')
    else:
        ax.hist(v, bins=12, color=color, edgecolor='white')
        ax.set_ylabel('cells')
    if len(v):
        ax.axvline(np.median(v), color='0.25', lw=1.2, zorder=4, label='median')
    ax.set_xlabel(xlabel)

if len(wc):
    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    quality_panel(axes[0], wc['max_seal_resistance'] / 1e9, 'peak seal resistance (GΩ)', FUNNEL_COLORS[0])
    axes[0].axvline(1.0, color=BAD, ls='--', lw=1, label='1 GΩ')
    axes[0].set_title('Seal quality')
    quality_panel(axes[1], wc['access_resistance'] / 1e6, 'access resistance (MΩ)', FUNNEL_COLORS[1])
    axes[1].set_title('Access resistance')
    quality_panel(axes[2], wc['holding_current'] / 1e-12, 'holding current (pA)', FUNNEL_COLORS[2])
    axes[2].set_title('Holding current')
    for ax in axes:
        if ax.get_legend_handles_labels()[0]:
            ax.legend(fontsize=8)
    plt.tight_layout(); plt.show()
    print('median peak seal R :', si(wc['max_seal_resistance'].median(), 'Ω'))
    print('median access R    :', si(wc['access_resistance'].median(), 'Ω'))
    print('median holding I   :', si(wc['holding_current'].median(), 'A'))
else:
    print('No whole-cell recordings in this dataset.')

## 10. Per-attempt drill-down

One row per attempt with outcome and key numbers. Sort/filter as needed — e.g.
`table[table['found_cell'] & ~table['broke_in']]` to inspect attempts that found
a cell but never reached whole-cell.

In [ ]:
table = df[['cell_dir', 'device', 'attempt_index', 'best_stage_name', 'outcome',
            'duration_s', 'max_seal_resistance', 'access_resistance',
            'holding_current', 'n_test_pulses']].copy()
table['duration_min'] = (table.pop('duration_s') / 60.0).round(1)
for col, unit in [('max_seal_resistance', 'Ω'), ('access_resistance', 'Ω'), ('holding_current', 'A')]:
    table[col] = table[col].map(lambda v: si(v, unit))
table

## 11. Full timeline per folder

One Gantt chart per analyzed folder: every device's patch states over wall-clock,
using **all** attempts (including the setup/cleaning idle that the rest of the
analysis excludes). The shaded span is the analyzed demo window — first to last
approached attempt — so you can see the idle before and after that is trimmed
from the active time.

In [ ]:
tl = am.state_timeline(raw_attempts)

# Analyzed demo window per folder = span of the approached attempts we keep.
window_by_folder = {}
for a in attempts:
    f = os.path.dirname(a.source)
    lo, hi = window_by_folder.get(f, (a.start_time, a.end_time))
    window_by_folder[f] = (min(lo, a.start_time), max(hi, a.end_time))

# Stable color per state across all folders.
states_seen = list(dict.fromkeys(tl['state']))
cmap = plt.get_cmap('tab20')
state_color = {s: cmap(i % 20) for i, s in enumerate(states_seen)}

for folder, g in tl.groupby('folder'):
    t0 = g['t_start'].min()
    devices = sorted(g['device'].unique())
    ypos = {d: i for i, d in enumerate(devices)}
    fig, ax = plt.subplots(figsize=(12, 0.6 * len(devices) + 2))
    if folder in window_by_folder:
        lo, hi = window_by_folder[folder]
        ax.axvspan((lo - t0) / 60.0, (hi - t0) / 60.0, color='0.5', alpha=0.15,
                   zorder=0, label='analyzed demo window')
    for _, r in g.iterrows():
        width = max((r['t_end'] - r['t_start']) / 60.0, 1e-3)
        ax.broken_barh([((r['t_start'] - t0) / 60.0, width)],
                       (ypos[r['device']] - 0.4, 0.8),
                       facecolors=state_color[r['state']], edgecolor='none')
    ax.set_yticks(range(len(devices)))
    ax.set_yticklabels(devices)
    ax.set_ylim(-0.6, len(devices) - 0.4)
    ax.set_xlabel('minutes since first event in folder')
    ax.set_title(f'{os.path.basename(folder)}  —  shaded = analyzed demo window')
    ax.grid(True, axis='x', alpha=0.25)
    handles = [plt.Rectangle((0, 0), 1, 1, color=state_color[s]) for s in states_seen]
    ax.legend(handles, states_seen, ncol=min(len(states_seen), 6), fontsize=7,
              loc='upper center', bbox_to_anchor=(0.5, -0.3), frameon=False)
    plt.tight_layout()
    plt.show()